# Snowflake Objetcs

<img src="Image1_ObjHrcy.png"  />

Organization : It is the top most object of Snowflake, that handles the bills, accounts and data failover cases( to retrieve data after the time travel period)
- It simplify account management and billing, replication and failover/ failback, snowflake data sharing and other account administrative tasks. 
- It also allows organization administration to view, create and manage all your accounts across different regions and cloud platforms.


Account: It comes below Organization
- Account Scalibility : Anyone can have multiple accounts
- Unique Access: Each account has its own unique URL
- Infrastructure: Each account is deployed on a single Cloud Service Provider(CSP) either AWS, GCP or Microsoft Azure
- Geography: Each account exists in a single geographic region
- Edition: Each account exists in single Snowflake edition : either Standard or Enterprise
- Account Usage Tracking: every account comes with a built in read only database called Snwofalke

It contains the ACCOUNT_USAGE schema. This is where you find views to track how many credits you've spent, who logged in, and which tables are taking up the most storage.

Follow the 05_Snowflake_Objects.sql file to look for the queries executed.

Stages: It is a platform which helps loading and unloading data into Snowflake. It is a location where data files are stored.
Stages can be internal or external. Internal stages are managed by Snowflake, while external stages are managed by the user and can be located in cloud storage services like AWS S3, Azure Blob Storage, or Google Cloud Storage.

Schema : In Snowflake, a schema is a logical container that holds database objects such as tables, views, and other structures. It helps organize and manage data within a database. Each schema belongs to a specific database and can contain multiple objects, allowing for better organization and access control. Tables and views cannot float in database, it should always be created within a schema. Schemas also provide a way to manage permissions and access to the objects they contain, enabling users to control who can read, write, or modify the data.

MANAGED ACCESS SCHEMA: In Snowflake, a Managed Access Schema is a schema where the schema owner (or a role with the appropriate grant-management privileges) controls object-level grants, rather than individual object owners managing access independently.

In a regular schema, the owner of a table or view can generally grant privileges on that object to other roles. In a managed access schema, object owners cannot independently grant or revoke privileges on their objects; grant management is centralized.


TABLES
| TABLE TYPE | PERSISTENCE | TIME TRAVEL | FAIL-SAFE | USE CASE |
|---|---|---|---|---|
| PERMANENT | Persistent until dropped | Upto 90 days | 7 days | Long term data storage |
| Temporary | Session Duration | None | None | Short Term Data Processing |
| Transient | Persistent Until dropped | 0-1 Day | None | Intermediate or Staging Data|
| External | Data Stored Externally | None | None | Querying data in external storage |
| Dynamic | Virtual, data generated on the fly | None | None | Real time data generation |
| Hybrid | Combines local and external data | Varies | Varies | Flexible Data Management |
| Iceberg | Persistent, open table format | Configurable | Configurable | large scale data analytics |

TRANSIENT TABLE: Snowflake supports creating Transient tables that persists until explicitly dropped and are available to all users with the apppropriate privileges.

They are similar to permanent tables with the key difference that they do not have a fail safe period and time travel upto 1 day.


EXTERNAL Table: It is a Snowflake feature that you can use to query data stored in an external stage as if the data were inside a table in snowwflake. The external stage is not part of Snowflake. So, Snowflake deosn't store or manage the stage.

External tables enables snowflake to store file level metadata including file names, verions and related properties.

It can access data stored in any format that the copy command supports except XML. External tables are read only. So, you can't perform DML operations.

Dynamic Table: They are automatically refreshed based on defined query and target freshness, simplifying data transformation and pipeline managament without requiring manual updates or custom scheduling

When to use?
- materialize query results without writing custom code
- avoid manual tracking data dependencies and refresh schedlue
- chain together multiple tables for data transformation in a pipeline.

Hybrid Table:
- optimized for low latency and high throughput (latency : time taken for the data from src to tgt, throughput: amt of data that can flow in unit time) using index based random reads and writes.
- It provides row based storage engine that supports row locking for high concurrency.
- enforce unique and referential integrity constraints, critical for transactional workloads.( referential integrity: every foreign key must be mapped to primary key of some other table)
- It leverages a row store as the primary data store to provide excellent operational query performance. When you write to a hybrid table, the data is written directly into the row store.
- Some data maybe cached into columnar format in order to provide better performance for analytical queries.


<img src="Image2_hyb_tbl.png"  />

Apache Iceberg tables:
- It combines the performance and query semantics of typical snowflake tables with external cloud storage that you manage.
- They are ideal for existing data lakes that you cannot or chose not to store in snowflake.
- It uses the Apache Iceberg open table format specification, which provides an abstraction layer on data files stored in open formats.
- You are responsible for all management of external cloud storage location, incluidng the configuration of data protection and recovery. But Snowflake does not provide <font color= red >Fail-safe storage </font> for Iceberg tables.

|ICEBERG TABLE | EXTERNAL TABLE|
|---|---|
|Snowflake uses the metadata file to know which file to read before touching the s3 bucket| It looks the folder, scans the folder to look up for the file|
|Much faster | little slow|
|you can perform DML | You cannot perform DML |

There is an <font color=green>Iceberg catalog</font> which enables a compute engine to manage and load Iceberg tables

It doesn't just store data. It organizes it into three distinct layers of files on S3:
- Data files: These are the actual records stored in Parquet format.
- Manifest files: These acts as an index. They track which data files belong to a specific version of the table and store min/max stats for each column(used for data pruning)
- Metadata files: This is the 'Source of truth'. It tracks table's schema, partitioning and snapshots. when an iceberg table is created, you have to choose iceberg catalog. This is the brain that tracks where the latest metadata file is located.


# Why Iceberg

- No vendor lock in : Even if you leave snowflake, your data is already in S3 in standard format. You don't have to export it.
- Interoperrability : You can query the same s3 filess with Snowflake for BI and Spark for machine learning at the same time.
- Performance: Unlike standard external tables, Iceberg table uses the metadata layer to skip unneccesary files making them nearly as fast as native Snowflake tables.

# Views

Snowflake support two types of views :
- Materialized views
- Non-materialized views(views)

Secure views: These have advantages over standard views including improved data privacy and data sharing. However, they can have performance impacts to take into consideration.

Snowflake recommends creating Materialized views when all 3 satisfy
- The query results doesn't change frequently.
- The query results are used very frequently
- The query consumes a lot of resources.

LIMITATION OF MATERIALIZED VIEWS:
1. They are highly optmized and can query one table at a time and so no joins can be used into its defintion
2. It doesn't support volatile functions like currentdate,currenttimestamp,etc.

# User Defined Functions (UDF)

- It allows us to perform operations that are not available through the built-in system defined functions.
- Create UDF whenever there is need to repeat functionality.
- Supports 4 language for UDF: Python, Java, SQl, Java Script
- Snowflake UDFs can return scalar quantity and tabular results as well.
- It supports overloading i.e functions with same name but different parameters.

# Store Procedures
- you can write procedural codes that includes sql statements, conditional statements, looping statements and cursors.
- It supports 5 languages: Snowfalke scripting(sQL), Java Script, Java, Python, Scala
- You can return a single value as well as a table
- Supports branching and looping
- Support dynamic sqls and execute it

# Difference between Store Procedure and UDF

| Store Procedure | UDF | 
|--- | --- |
| It may or may not return results | It must always return results |
| These are called as independent statements | These are called in sql statements |
| Values returned by Procedures cannot be reused in SQL statements | Values returned by functions can be directly used in SQL statements|
| Single procedure per CAll statement | Single SQl statement can call multiple functions|
| Procedures can execute DDL and DML functions | It doesn't have access to perform database operations|

# FILE FORMAT

Syntax
Create or alter FILE FORMAT

-- CSV

Compression = auto | GZIP | Snappy | None
Record_delimiter ='<string>'
Field_delimiter='<string>'
File_extention='<string>'
Skip_header=<integer>
Timestamp_format='<string>'|AUTO
TRIM_SPACE =TRUE | FALSE
FIELD_OPTIONALLY_ENCLOSED_BY = '<CHARACTER>' | NONE
NULL_IF =('<STRING>','<STRING>')
EMPTY_FIELD_AS_NULL = TRUE | FALSE
ERROR_ON_COLUMN_COUNT_MISMATCH = TRUE | FALSE

-- JSON

COMPRESSION = AUTO |GZIP | NONE
DATA_FORMAT = '<STRING>' | AUTO
TIMESTAMP_FORMAT = '<STRING>' | AUTO
STRIP_OUTER_ARRAY = TRUE | FALSE
STRIP_NULL_VALUES = TRUE | FALSE
IGNORE_UTF8_ERRORS = TRUE | FALSE
REPLACE_INVALID_CHARACTERS = TRUE | FALSE

-- PARQUET
COMPRESSION = AUTO |GZIP | NONE
BINARY_AS_TEXT = TRUE | FALSE
USE_LOGICAL_TYPE = TRUE | FALSE
TRIM_SPACE = TRUE | FALSE
NULL_IF =('<STRING>','<STRING>')

# SNOWPIPE

It is Snowflake's automated, continuous data ingestion service. 
Unlike traditional bulk loading (that uses COPY COMMAND), Snowpipe loads data in micro - batches as soon as it arrives in your cloud storage(S3, Azure BON)

-- Characteristics

- Serverless : You don't need to manage a virtual warehous. Snowflake provides and manages the computing power automatically.
- Micro- Batching: It is designed to load small amounts of data frequently, making data available for analysis in minutes rather than hours
- Cost Efficient: You are charged based on the actual compute time used to process the files. rather than a running warehouse hourly rate.

# Shares

These are named Snowflake objects that encapsulate all of the information required to share a database. Data providers add snowflake objects ( database, shemas, tables, secure views,etc) to a share using either or both of the following options:
- Grant privileges on objects to a share via a database role
- Grant privileges on objetcs directly to a share

Shares are secure, configurable and controlled completey by the provider account

- New objects added to a share are completely availble to all consumers providing real-time accesss to imported data
- Updates to existing objects in a share become immediately avaible to all consumers
- access to a share(or any objetc in a share) can be revoked at any time.

# Sequence
It is used to generate unique numbers across sessions and statements, including concurrent statements. They can be used to generate values for a primary key or any column that requires a unique value.

# SNOWFLAKE ML

YOU CAN create end to end ML workflows in SNowflake.
- prepare the data
- cerate and use features with snowflake feature store
- train models with CPUs or GPUs using any open-source prackage from Snowflake Notebooks on Container Runtime
- create experiments to evaluate your trained models against set metrics.
- Operationalize your pipelines using Snoflake ML jobs
- Deploy your model for inference at scale with Snowflake Model registry
- Monitor your production models with ML observability and exolainability

- you can deploy models developed in Snowflake outside of snowflake and externally trained models can be brought into snowflake for inference

# Applications

- Web based Snowflake interface : Snowsight
- Command line clients : Snowflake CLI, SnowSQL (CLI client)
- Extention : Snowflake extention for VS code
- 3rd party integration: SNowflake conector
- 3rd party software: Snowflake ecosystem
